In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler, TensorDataset
from torch.nn.utils import clip_grad_norm_
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import DebertaTokenizer, DebertaForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm import tqdm
import numpy as np
import random
import os

In [ ]:
# Ensure deterministic behavior
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)


In [2]:
# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [ ]:
# Load 20 Newsgroups dataset
newsgroups_data = fetch_20newsgroups(subset='train')


In [ ]:
# Filter out empty texts
data = [(text, label) for text, label in zip(newsgroups_data.data, newsgroups_data.target) if text.strip() != ""]
texts, labels = zip(*data)

In [ ]:
# Assign directly to train_texts and train_labels
train_texts = [text for text in newsgroups_data.data if text.strip() != ""]
train_labels = [label for text, label in zip(newsgroups_data.data, newsgroups_data.target) if text.strip() != ""]


In [ ]:

# Load DeBERTa tokenizer
tokenizer = DebertaTokenizer.from_pretrained("microsoft/deberta-base")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Tokenize in chunks to reduce memory usage
max_length = 512
batch_size = 500
input_ids, attention_masks = [], []
for i in range(0, len(train_texts), batch_size):
    batch = tokenizer(train_texts[i:i+batch_size], padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    input_ids.append(batch['input_ids'])
    attention_masks.append(batch['attention_mask'])

In [ ]:
train_input_ids = torch.cat(input_ids, dim=0)
train_attention_mask = torch.cat(attention_masks, dim=0)
train_labels_tensor = torch.tensor(train_labels, dtype=torch.long)


In [ ]:
# Dataset and Dataloader
train_dataset = TensorDataset(train_input_ids, train_attention_mask, train_labels_tensor)
train_dataloader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=8, num_workers=0)


In [ ]:
# Load DeBERTa model
model = DebertaForSequenceClassification.from_pretrained("microsoft/deberta-base", num_labels=20)
model.to(device)


Some weights of DebertaForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DebertaForSequenceClassification(
  (deberta): DebertaModel(
    (embeddings): DebertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=0)
      (LayerNorm): DebertaLayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaLayer(
          (attention): DebertaAttention(
            (self): DisentangledSelfAttention(
              (in_proj): Linear(in_features=768, out_features=2304, bias=False)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (pos_proj): Linear(in_features=768, out_features=768, bias=False)
              (pos_q_proj): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): DebertaLayerNorm()
              (dropout): Dropout(p=0

In [ ]:
# Optimizer and Scheduler
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
epochs = 3
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)


In [ ]:
total_steps

4245

In [ ]:
# Training loop
def train():
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

        for batch in progress_bar:
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)

            model.zero_grad()
            outputs = model(input_ids=b_input_ids, attention_mask=b_input_mask, labels=b_labels)
            loss = outputs.loss
            total_loss += loss.item()

            loss.backward()
            clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            progress_bar.set_postfix({"loss": loss.item()})

        avg_loss = total_loss / len(train_dataloader)
        print(f"Epoch {epoch+1} finished. Average Loss: {avg_loss:.4f}")

print(" Setup complete. Run train() to begin training.")


 Setup complete. Run train() to begin training.


In [ ]:
# Run training safely
if __name__ == "__main__":
    train()


Epoch 1 finished. Average Loss: 0.8287


Epoch 2 finished. Average Loss: 0.2746


Epoch 3 finished. Average Loss: 0.1366


In [ ]:
# Save path
save_directory = "./saved_deberta_model"

# Create folder if it doesn't exist
import os
os.makedirs(save_directory, exist_ok=True)

# Save model and tokenizer
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"✅ Model and tokenizer saved to: {save_directory}")


✅ Model and tokenizer saved to: ./saved_deberta_model


In [ ]:
import shutil
shutil.make_archive('deberta_model', 'zip', './saved_deberta_model')

from google.colab import files
files.download("deberta_model.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
from transformers import DebertaTokenizer, DebertaForSequenceClassification
import torch
import zipfile
import os


# Set paths
zip_path = "/content/drive/MyDrive/deberta_model.zip"
extract_path = "/content/deberta_model"

# Unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Nested folder correction
model_path = "/content/deberta_model/deberta_model"

# Load model and tokenizer (assumes model.safetensors is present)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
deberta_model = DebertaForSequenceClassification.from_pretrained(model_path, use_safetensors=True)
deberta_tokenizer = DebertaTokenizer.from_pretrained(model_path)
deberta_model.to(device)

#print("DeBERTa model and tokenizer loaded from nested folder.")


cuda


DebertaForSequenceClassification(
  (deberta): DebertaModel(
    (embeddings): DebertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=0)
      (LayerNorm): DebertaLayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaLayer(
          (attention): DebertaAttention(
            (self): DisentangledSelfAttention(
              (in_proj): Linear(in_features=768, out_features=2304, bias=False)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (pos_proj): Linear(in_features=768, out_features=768, bias=False)
              (pos_q_proj): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): DebertaLayerNorm()
              (dropout): Dropout(p=0

In [5]:
from sklearn.datasets import fetch_20newsgroups

test_data = fetch_20newsgroups(subset='test')
test_texts = test_data.data
test_labels = test_data.target




In [6]:
# Tokenize test data
test_encodings = deberta_tokenizer(test_texts, padding=True, truncation=True, max_length=512)
test_input_ids = torch.tensor(test_encodings['input_ids'])
test_attention_mask = torch.tensor(test_encodings['attention_mask'])
test_labels_tensor = torch.tensor(test_labels)


In [7]:
# Dataset and Dataloader
test_dataset = TensorDataset(test_input_ids, test_attention_mask, test_labels_tensor)
test_dataloader = DataLoader(test_dataset, batch_size=8, sampler=SequentialSampler(test_dataset))


In [8]:
# Evaluation function for deberta
def evaluate_test_deberta():
    deberta_model.eval()
    preds = []
    true_labels = []

    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Evaluating"):
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)

            outputs = deberta_model(b_input_ids, attention_mask=b_input_mask)
            logits = outputs.logits

            preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
            true_labels.extend(b_labels.cpu().numpy())

    acc = accuracy_score(true_labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, preds, average='weighted')

    print("\n Evaluation Completed")
    print(f"Test Accuracy: {acc:.4f}")
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")

print(" Setup complete. Run evaluate_test() to evaluate your model.")


 Setup complete. Run evaluate_test() to evaluate your model.


In [9]:
evaluate_test_deberta()

Evaluating: 100%|██████████| 942/942 [06:15<00:00,  2.51it/s]


 Evaluation Completed
Test Accuracy: 0.8662
Precision: 0.8700, Recall: 0.8662, F1 Score: 0.8665
